# Week 9: Transformers and Contextual Representations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/09/Week_09_Transformers_Contextual_Representations.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)


## Learning goals

By the end of this session, you should be able to:

- Explain how self-attention differs from Week 8 attention pooling.
- Describe queries, keys, and values as roles in attention.
- Explain why transformers need positional information.
- Train a tiny transformer-style classifier.
- Visualize attention heads as representation-building clues.
- Compare static token embeddings with contextual token representations.
- Interpret transformer attention carefully, without treating it as a perfect explanation.

**Course habit:** inspect representation -> change one thing -> run -> observe -> explain.

**New representation habit:** ask how context changes a token representation.


---

## Environment

**Dependencies:** `torch`, `numpy`, `matplotlib`, `scikit-learn`. CPU is enough for the full notebook.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/09/Week_09_Transformers_Contextual_Representations.ipynb
```

### Colab

1. Open the notebook via the badge above.
2. Runtime -> Change runtime type -> CPU is fine.
3. Run cells in order.

This notebook uses tiny inline datasets. There are no dataset downloads and no Hugging Face dependency in the core path.


In [ ]:
import math
import random
import re
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from sklearn.decomposition import PCA

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(9)


---

## 1. From attention pooling to self-attention

Week 8 used attention pooling:

```text
word embeddings -> attention weights -> one sentence representation -> prediction
```

That helped us ask: **which tokens mattered for the sentence decision?**

Transformers use self-attention differently:

```text
each token representation -> looks at other tokens -> updated contextual token representation
```

The key change is that attention now updates **every token**, not only the final sentence summary.

**Pause and predict**

1. Why might `bank` need a different representation in `bank approves loan` and `bank has mud near river`?
2. Why does a transformer need position information if it sees all tokens at once?
3. What would count as evidence that a token representation became contextual?


### External anchors for this week

These are optional references, not required dependencies. Use them as visual anchors before or after class.

| Chapter | Resource | Why it helps |
|---|---|---|
| Self-attention intuition | [3Blue1Brown: Attention in transformers, visually explained](https://www.youtube.com/watch?v=eMlx5fFNoYc) | Best first visual explanation of attention as moving information between token vectors. |
| Transformer walkthrough | [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) | Interactive GPT-2 view of tokenization, embeddings, attention, MLP blocks, and output probabilities. |
| Architecture overview | [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) | Clear visual bridge from attention to the full transformer block. |
| Contextual embeddings | [The Illustrated BERT](https://jalammar.github.io/illustrated-bert/) | Shows why contextual token representations matter for pretrained models. |
| Code reference | [The Annotated Transformer](http://nlp.seas.harvard.edu/annotated-transformer/) | Optional code-oriented reference for students who want implementation detail. |
| Attention inspection | [BertViz](https://github.com/jessevig/bertviz) | Optional tool for visualizing attention heads in pretrained transformer models. |
| Deep coding extension | [Andrej Karpathy: Let's build GPT from scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY) | Long optional coding deep dive; useful only after the core notebook feels comfortable. |
| Cautionary reading | [Attention is not Explanation](https://arxiv.org/abs/1902.10186) | Reminder that attention maps are useful evidence, not guaranteed human explanations. |
| Original paper | [Attention Is All You Need](https://arxiv.org/abs/1706.03762) | Historical reference; optional and math-heavy. |


---

## 2. Self-attention intuition

A self-attention layer gives each token three learned roles:

| Role | Intuition |
|---|---|
| Query | what this token is looking for |
| Key | what this token offers for matching |
| Value | what information this token contributes if selected |

A token compares its query with other tokens' keys. The resulting scores become attention weights. The values are then mixed according to those weights.

Minimal version:

```text
attention weights = softmax(query dot key scores)
updated token = weighted average of values
```

We will not dwell on derivations. The important representation idea is: **tokens update themselves by borrowing information from other tokens.**


In [ ]:
def draw_self_attention_sentence(tokens, focus_index=1):
    fig, ax = plt.subplots(figsize=(max(7, len(tokens) * 1.2), 2.4))
    ax.axis("off")
    ax.set_xlim(-0.6, len(tokens) - 0.4)
    ax.set_ylim(-0.9, 1.0)
    for i, tok in enumerate(tokens):
        color = "#ffd166" if i == focus_index else "#d8ecff"
        ax.text(i, 0, tok, ha="center", va="center", fontsize=12,
                bbox=dict(boxstyle="round,pad=0.35", facecolor=color, edgecolor="#4c78a8"))
        if i != focus_index:
            ax.annotate("", xy=(focus_index, 0.18), xytext=(i, 0.18),
                        arrowprops=dict(arrowstyle="->", lw=1.2, alpha=0.75))
    ax.set_title(f"Self-attention idea: '{tokens[focus_index]}' can look at the other tokens")
    plt.show()

draw_self_attention_sentence(["[CLS]", "bank", "approves", "loan"], focus_index=1)


---

## 3. Tiny ambiguous-word dataset

We use a small `bank` dataset.

The word `bank` appears in both classes:

- finance context: `loan`, `money`, `account`, `savings`
- river context: `river`, `water`, `mud`, `boat`

The task is deliberately small. Its purpose is to make contextual representations inspectable, not to build a serious language model.


In [ ]:
finance_sentences = [
    "bank approves loan",
    "bank manages savings",
    "bank account earns interest",
    "deposit money at bank",
    "cash teller works at bank",
    "credit card from bank",
    "bank invests client money",
    "loan officer calls from bank",
    "bank transfers cash today",
    "customer opens bank account",
    "bank charges account fee",
    "savings grow inside bank",
]

river_sentences = [
    "bank has mud near river",
    "river water touches bank",
    "fish swim by river bank",
    "trees grow on bank near water",
    "boat rests beside bank",
    "stream curves around bank",
    "bank erodes after rain",
    "ducks walk along river bank",
    "muddy bank borders river",
    "grass covers the river bank",
    "water rises over bank",
    "stone path follows bank",
]

labels = ["finance", "river"]
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

all_examples = [(s, label_to_idx["finance"]) for s in finance_sentences]
all_examples += [(s, label_to_idx["river"]) for s in river_sentences]
random.shuffle(all_examples)

TOKEN_RE = re.compile(r"[a-z]+")

def tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter = Counter()
for text, _ in all_examples:
    counter.update(tokenize(text))

special_tokens = ["<PAD>", "<UNK>", "[CLS]"]
word_to_idx = {tok: i for i, tok in enumerate(special_tokens)}
for word in sorted(counter):
    word_to_idx[word] = len(word_to_idx)
idx_to_word = {i: word for word, i in word_to_idx.items()}
pad_idx = word_to_idx["<PAD>"]
cls_idx = word_to_idx["[CLS]"]

print("examples:", len(all_examples))
print("vocabulary size:", len(word_to_idx))
print("labels:", labels)
print("sample tokens:", tokenize(all_examples[0][0]))


In [ ]:
def show_dataset_examples(n=8):
    for text, y in all_examples[:n]:
        print(f"{text:34s} -> {idx_to_label[y]}")

show_dataset_examples()


---

## 4. A tiny transformer-style classifier

The model has five important parts:

```text
token ids
  -> token embeddings
  -> + positional embeddings
  -> self-attention block
  -> contextual [CLS] representation
  -> classifier
```

The `[CLS]` token is a learned summary position. After self-attention, its contextual representation is used for the final prediction.

This is tiny compared with real transformers, but the representation logic is the same.


In [ ]:
def encode_sentence(text):
    ids = [cls_idx]
    ids += [word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in tokenize(text)]
    return torch.tensor(ids, dtype=torch.long)


def collate_sentences(batch):
    encoded = [encode_sentence(text) for text, _ in batch]
    lengths = torch.tensor([len(x) for x in encoded], dtype=torch.long)
    y = torch.tensor([label for _, label in batch], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.full((len(batch), max_len), pad_idx, dtype=torch.long)
    for i, item in enumerate(encoded):
        x[i, : len(item)] = item
    return x, lengths, y

train_loader = DataLoader(all_examples, batch_size=8, shuffle=True, collate_fn=collate_sentences)

class TinyTransformerBlock(nn.Module):
    def __init__(self, d_model=32, num_heads=2, ff_dim=64, dropout=0.0):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None, return_attention=False):
        attn_out, attn_weights = self.attn(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        if return_attention:
            return x, attn_weights
        return x

class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=32, num_heads=2, ff_dim=64, max_len=16):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        self.block = TinyTransformerBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, return_attention=False, return_representations=False):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        static = self.token_embedding(x)
        h0 = static + self.pos_embedding(positions)
        key_padding_mask = x == pad_idx
        h, attn = self.block(h0, key_padding_mask=key_padding_mask, return_attention=True)
        logits = self.classifier(h[:, 0])
        if return_representations:
            return logits, attn, static, h0, h
        if return_attention:
            return logits, attn
        return logits


def accuracy_from_logits(logits, y):
    return (logits.argmax(dim=1) == y).float().mean().item()


def train_transformer(d_model=32, num_heads=2, ff_dim=64, epochs=120, lr=0.01, seed=9):
    if d_model % num_heads != 0:
        raise ValueError("d_model must be divisible by num_heads")
    set_seed(seed)
    model = TinyTransformerClassifier(
        vocab_size=len(word_to_idx),
        num_classes=len(labels),
        d_model=d_model,
        num_heads=num_heads,
        ff_dim=ff_dim,
        max_len=16,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": []}

    for epoch in range(epochs):
        model.train()
        losses, accs = [], []
        for x, lengths, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            accs.append(accuracy_from_logits(logits, y))
        history["loss"].append(float(np.mean(losses)))
        history["accuracy"].append(float(np.mean(accs)))
    return model, history


def plot_training_history(history, title="Tiny transformer training"):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(history["loss"])
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[1].plot(history["accuracy"])
    axes[1].set_title("Training accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylim(0, 1.05)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

transformer_model, transformer_history = train_transformer()
plot_training_history(transformer_history)
print("final training accuracy:", round(transformer_history["accuracy"][-1], 3))


### What to inspect

- `token_embedding`: static vector for each token.
- `pos_embedding`: learned vector for each position.
- `block.attn`: self-attention across tokens.
- `block.ff`: feed-forward refinement after attention.
- `classifier`: maps the contextual `[CLS]` representation to the label.

**Common confusion:** this is not Week 8 attention pooling. Self-attention produces updated token representations for all tokens.


In [ ]:
@torch.no_grad()
def model_details(model, sentence):
    model.eval()
    x, lengths, _ = collate_sentences([(sentence, 0)])
    x = x.to(device)
    logits, attn, static, h0, contextual = model(
        x, return_attention=True, return_representations=True
    )
    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()
    pred = int(probs.argmax())
    tokens = ["[CLS]"] + tokenize(sentence)
    return {
        "tokens": tokens,
        "probs": probs,
        "pred": pred,
        "attn": attn.squeeze(0).cpu(),
        "static": static.squeeze(0).cpu(),
        "h0": h0.squeeze(0).cpu(),
        "contextual": contextual.squeeze(0).cpu(),
    }


def predict_sentence(model, sentence):
    info = model_details(model, sentence)
    pred_label = idx_to_label[info["pred"]]
    print(f"{sentence!r} -> {pred_label} | p(finance)={info['probs'][0]:.2f} p(river)={info['probs'][1]:.2f}")
    return info


def plot_cls_attention(model, sentence, title=None):
    info = model_details(model, sentence)
    tokens = info["tokens"]
    attn = info["attn"].numpy()  # heads, target_tokens, source_tokens
    cls_attention = attn[:, 0, : len(tokens)]

    fig, ax = plt.subplots(figsize=(max(7, len(tokens) * 0.9), 2.2 + 0.35 * cls_attention.shape[0]))
    im = ax.imshow(cls_attention, cmap="YlOrRd", aspect="auto", vmin=0, vmax=max(0.45, float(cls_attention.max())))
    ax.set_xticks(range(len(tokens)), labels=tokens, rotation=30, ha="right")
    ax.set_yticks(range(cls_attention.shape[0]), labels=[f"head {i}" for i in range(cls_attention.shape[0])])
    pred_label = idx_to_label[info["pred"]]
    ax.set_title(title or f"[CLS] attention | prediction: {pred_label}")
    for i in range(cls_attention.shape[0]):
        for j in range(len(tokens)):
            ax.text(j, i, f"{cls_attention[i, j]:.2f}", ha="center", va="center", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    plt.show()


def find_token_index(tokens, token):
    for i, tok in enumerate(tokens):
        if tok == token:
            return i
    raise ValueError(f"{token!r} not found in {tokens}")


def bank_vectors(model, sentence):
    info = model_details(model, sentence)
    bank_idx = find_token_index(info["tokens"], "bank")
    return info, info["static"][bank_idx], info["contextual"][bank_idx]


def compare_bank_contexts(model, sentence_a="bank approves loan", sentence_b="bank has mud near river"):
    info_a, static_a, contextual_a = bank_vectors(model, sentence_a)
    info_b, static_b, contextual_b = bank_vectors(model, sentence_b)
    static_cos = F.cosine_similarity(static_a.unsqueeze(0), static_b.unsqueeze(0)).item()
    contextual_cos = F.cosine_similarity(contextual_a.unsqueeze(0), contextual_b.unsqueeze(0)).item()
    print("Sentence A:", sentence_a)
    print("Sentence B:", sentence_b)
    print(f"Static token-embedding cosine for 'bank':      {static_cos:.3f}")
    print(f"Contextual representation cosine for 'bank': {contextual_cos:.3f}")
    print("\nInterpretation: the token embedding is shared, but self-attention can move the contextual representation apart.")


def plot_contextual_bank_pca(model, examples=None, title="Contextual representations of 'bank'"):
    if examples is None:
        examples = all_examples
    vectors, colors, texts = [], [], []
    for text, y in examples:
        if "bank" not in tokenize(text):
            continue
        info, _, contextual = bank_vectors(model, text)
        vectors.append(contextual.numpy())
        colors.append(y)
        texts.append(text)
    coords = PCA(n_components=2, random_state=0).fit_transform(np.array(vectors))
    plt.figure(figsize=(7, 5))
    palette = {0: "#4c78a8", 1: "#f58518"}
    for label_idx, label in idx_to_label.items():
        mask = np.array(colors) == label_idx
        plt.scatter(coords[mask, 0], coords[mask, 1], label=label, color=palette[label_idx], s=55)
    for i, text in enumerate(texts):
        short = " ".join(tokenize(text)[:3])
        plt.text(coords[i, 0] + 0.03, coords[i, 1] + 0.03, short, fontsize=8)
    plt.title(title)
    plt.xlabel("PCA component 1")
    plt.ylabel("PCA component 2")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
for sentence in ["bank approves loan", "bank has mud near river"]:
    predict_sentence(transformer_model, sentence)
    plot_cls_attention(transformer_model, sentence)


**Pause and reflect**

1. Which context words does `[CLS]` attend to?
2. Do the heads look identical or different?
3. Does the attention map prove the model's reasoning, or only give us evidence about this model's behavior?


---

## 5. Coding block 1: change the tiny transformer

**Goal:** edit one or two model settings, train briefly, and inspect attention.

**Core path**

1. Change `d_model`, `num_heads`, `epochs`, or `lr`.
2. Re-run training.
3. Compare attention maps for one finance and one river sentence.
4. Explain what changed.

Keep `d_model` divisible by `num_heads`.


In [ ]:
# TODO: edit one or two values, then re-run.
student_transformer_config = {
    "d_model": 24,
    "num_heads": 2,
    "ff_dim": 48,
    "epochs": 90,
    "lr": 0.01,
}

student_transformer, student_history = train_transformer(**student_transformer_config)
plot_training_history(student_history, "Student transformer experiment")

for sentence in ["bank approves loan", "bank has mud near river"]:
    predict_sentence(student_transformer, sentence)
    plot_cls_attention(student_transformer, sentence)


### Transformer experiment report prompt

Write 4-6 sentences:

1. What did you change?
2. What happened to training accuracy or loss?
3. Did attention become sharper, more diffuse, or similar?
4. Which context words did `[CLS]` attend to?
5. What does this suggest about the learned representation?


---

## 6. Contextual representations

A static embedding table stores one vector per token. The token `bank` has one static embedding.

After self-attention, the contextual representation of `bank` can differ across sentences.

That is the transformer representation-learning idea in one sentence:

> Same token, different context, different representation.


In [ ]:
compare_bank_contexts(transformer_model)
plot_contextual_bank_pca(transformer_model)


**Pause and reflect**

1. Why is the static embedding cosine close to 1?
2. Why can the contextual cosine be lower?
3. What information from the sentence might move the `bank` representation?


---

## 7. Attention heads as inspection tools

Multi-head attention lets the model learn several attention patterns in parallel.

In a large transformer, heads can sometimes specialize. In our tiny model, heads may also be redundant, unstable, or surprising.

Read attention maps as debugging evidence:

- useful for inspection
- useful for comparison
- not guaranteed human explanations


In [ ]:
custom_examples = [
    "bank transfers cash today",
    "water rises over bank",
    "bank near water",
    "bank near money",
]

for sentence in custom_examples:
    predict_sentence(transformer_model, sentence)
    plot_cls_attention(transformer_model, sentence)


---

## 8. Coding block 2: custom contextual analysis

**Goal:** test custom sentences and inspect both attention and contextual representations.

**Core path**

1. Write at least two finance-like and two river-like sentences.
2. Compare predictions.
3. Plot attention maps.
4. Compare contextual `bank` vectors.
5. Explain what the model seems to use.

Use words from the tiny vocabulary when possible. Unknown words become `<UNK>`.


In [ ]:
# TODO: change these examples.
student_sentences = [
    "bank account earns interest",
    "bank has mud near river",
    "bank near cash",
    "bank near water",
]

for sentence in student_sentences:
    predict_sentence(transformer_model, sentence)
    plot_cls_attention(transformer_model, sentence)

# TODO: choose two sentences that both contain the word "bank".
compare_bank_contexts(
    transformer_model,
    sentence_a="bank account earns interest",
    sentence_b="bank has mud near river",
)


### Contextual analysis report prompt

Write 5-7 sentences:

1. Which custom sentences were classified confidently?
2. Which tokens received high `[CLS]` attention?
3. Did different heads behave differently?
4. Did the contextual `bank` comparison support your interpretation?
5. Which example was ambiguous or misleading?
6. What is one limitation of this tiny transformer?


---

## 9. Assignment 4 preview

Assignment 4 will ask you to analyze transformer representations.

Minimum evidence will likely include:

- a controlled transformer experiment
- attention heatmaps for contrasting examples
- contextual representation comparison
- examples where context helped
- examples where attention was ambiguous or misleading
- a short explanation of self-attention vs Week 8 attention pooling

The main question is not "is this a big transformer?" The main question is: **how does self-attention change token representations using context?**


---

## Wrap-up: takeaways

1. Week 8 attention pooling built one sentence representation.
2. Transformer self-attention updates every token representation.
3. Token embeddings are static; contextual representations depend on surrounding tokens.
4. Positional embeddings help the model know token order.
5. Attention heads are useful inspection tools, but not perfect explanations.
6. Transformers build representations by repeatedly mixing and refining token information.


---

## Homework / Post-class Extensions

Optional unless assigned.

| Idea | What to try |
|------|-------------|
| One head vs two heads | Compare `num_heads=1` and `num_heads=2` |
| Larger representation | Try `d_model=48`, `num_heads=3` |
| Extra ambiguous word | Add `apple` as fruit/company examples |
| Full attention matrix | Plot every target token attending to every source token |
| Two transformer blocks | Stack a second `TinyTransformerBlock` and compare behavior |
| Pretrained bridge | Open The Illustrated BERT and compare with our tiny contextual vectors |
